# Stage B - Random Forest with GridSearchCV
Intermediate model using PCA features from flattened CIFAR-10 images.

In [ ]:
# Portable project setup. Works after cloning GitHub repo or extracting the ZIP.
import os, sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
# If Colab opens the notebook from another location, uncomment and edit this line:
# PROJECT_ROOT = Path("/content/MLFinalProject")
sys.path.insert(0, str(PROJECT_ROOT))
FIG_DIR = PROJECT_ROOT / "artifacts" / "figures"
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"
for d in [FIG_DIR, TABLE_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

import numpy as np, pandas as pd, json, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from src.data import get_preprocessed_flat, CLASSES, set_seeds
from src.metrics import compute_metrics, classification_report_df, confusion_matrix_array, per_class_accuracy
from src import viz
set_seeds(42)

In [ ]:
(X_train, y_train), (X_val, y_val), (X_test, y_test), stats = get_preprocessed_flat()
X_trainval = np.concatenate([X_train, X_val], axis=0)
y_trainval = np.concatenate([y_train, y_val], axis=0)
print("Train+val:", X_trainval.shape, "Test:", X_test.shape)

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=150, random_state=42)),
    ("rf", RandomForestClassifier(random_state=42, n_jobs=-1))
])
param_grid = {
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [20, None],
    "rf__max_features": ["sqrt", "log2"],
    "rf__min_samples_split": [2]
}
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring="accuracy", n_jobs=-1, verbose=2)
grid_search.fit(X_trainval, y_trainval)
cv_results = pd.DataFrame(grid_search.cv_results_).sort_values("rank_test_score")
display(cv_results[["param_rf__n_estimators","param_rf__max_depth","param_rf__max_features","mean_test_score","std_test_score","rank_test_score"]])
cv_results.to_csv(TABLE_DIR / "stageB_cv_results.csv", index=False)
print("Best params:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

In [ ]:
best_rf = grid_search.best_estimator_
joblib.dump(best_rf, MODEL_DIR / "stageB_random_forest.pkl")
y_pred = best_rf.predict(X_test)
metrics = compute_metrics(y_test, y_pred, model_name="Random Forest + PCA")
print(metrics)
report_df = classification_report_df(y_test, y_pred)
display(report_df)
report_df.to_csv(TABLE_DIR / "stageB_classification_report.csv")
cm = confusion_matrix_array(y_test, y_pred)
viz.plot_confusion_matrix(cm, title="Stage B Random Forest Confusion Matrix", save_name="stageB_confusion_matrix.png")

In [ ]:
result_record = {
    "model": "Random Forest + GridSearchCV",
    "stage": "B",
    "preprocessing": "flatten + StandardScaler + PCA(150)",
    "best_params": grid_search.best_params_,
    "best_cv_score": float(grid_search.best_score_),
    "test_accuracy": float(metrics["accuracy"]),
    "macro_f1": float(metrics["macro_f1"]),
    "weighted_f1": float(metrics["weighted_f1"])
}
with open(TABLE_DIR / "stageB_result.json", "w") as f: json.dump(result_record, f, indent=2)
print("Stage B result saved.")

## Stage B interpretation
Random Forest can learn non-linear combinations of PCA components, so it should improve over logistic regression. However, the input is still flattened image data, and PCA components do not preserve local image structure like edges and textures. This means the model is still weaker than a CNN for CIFAR-10.